# Python in the Smoke 💨 🐍 💨
## Applied notebook: image data with scikit-image, and the V&A's fashion plates

---

**Repo:** [`python-in-the-smoke`](https://markgotham.github.io/python-in-the-smoke)

**By:** Mark Gotham

**Licence:** MIT

**Colour and symbol key:** [Itemised and explained here](https://markgotham.github.io/python-in-the-smoke/#note-to-students)

**Assumed knowledge:**

This notebook builds on the foundational Part I notebooks (a to j).
No other applied notebook is required (though [apis_json](./apis_json.ipynb) is relevant to Part 3).
Each idea in the table below has appeared before, and now returns in a new setting.

| Part I | Assumed knowledge | How it appears here |
|---|---|---|
| a | numbers, text, f-strings | Formatted tables of results |
| b | lists, indices, slices, dictionaries | An image is a grid of numbers, and cropping is slicing; API results are lists of dictionaries |
| c | Booleans, comparisons | A comparison on every pixel at once gives a "mask" |
| d | loops | Looping over decades, search results and images |
| e | functions | `centre_crop`, `colour_summary`, `show_images` |
| f | packages, dataframes | `scikit-image` and `pandas`: tables of images and their measurements |
| g | files and paths | Saving downloaded images and a record of the sample |
| h | cleaning data | Parsing dates; dropping records that cannot be used |
| j | visualising data | `matplotlib` plots, here used to show images as well as numbers |

**New here, and explained where it appears: arrays** (Part 1).

Retrieving data from the web (APIs) is taught elsewhere: Part 3 only *applies* it.

---
## 🟢 Information: Why this topic, and why this design?

**The cultural question.**
In 1856, in London, an 18-year-old chemist called William Perkin made the first synthetic dye: a purple called *mauve*.
A popular story says that cheap synthetic dyes then changed what Victorian fashion looked like: more colour, and more purple.

Can we test a story like that with *images*?
Fashion plates (illustrations of the latest styles, printed in magazines) survive in large numbers,
and London's Victoria and Albert Museum (the V&A) makes thousands of them available online.
So we can ask: **did the colour of fashion plates change after 1856?**

📝 As so often, the question and method is at least as interesting as any answer we might reach.
How is a study like this built, and how easily it can mislead.

**The plan.**
1. **Part 1:** an image is just a grid of numbers. We use pictures that come built in to `scikit-image`.
2. **Part 2:** turn a reasonable (but informal) question like "how colourful is this?" into quantitative measurement,
and check in cases where we "know" the answer.
3. **Part 3:** download a real sample of fashion plates from the V&A, and look at them before measuring.
4. **Part 4:** measure every plate, compare decades, and then try hard to break our own result.
5. **Part 5:** a template for your own project, with ideas, sources and things to check.

After Part 5 you should be able to answer: "what would *my* research question be, and what would I need to check?"

---
## 🟢 Information: Set up

We need `scikit-image` (for images), `matplotlib` (for showing them) and `pandas` (tables).
- **Installing** uses the name `scikit-image`.
- **Importing** uses the name `skimage`.

📝 This is a bit like an app whose name on the store differs from the label on its icon.
It is a classic source of errors: `pip install skimage` is the wrong way round, and `import scikit-image` is a syntax error (we try it on purpose in the Common Errors section).

Run the cell below once. On Colab, these packages are usually already installed.

📝 We ask for `scikit-image` 0.25 or 0.26, because the outputs quoted in the solutions come from 0.26.
Another version may give slightly different results (that is itself a lesson).
Version 0.26 needs Python 3.11 or later; on Python 3.10 the installer will choose 0.25 for you.

In [ ]:
%pip install "scikit-image>=0.25,<0.27" matplotlib pandas

In [ ]:
import json
import math
import re
import time
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import skimage
from skimage import color, data, io, segmentation, filters

In [ ]:
print("scikit-image", skimage.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)

📝 **Record your versions.** Results depend on the versions of the packages, so write them down beside any findings.
They are printed above.

If an import fails, restart the kernel (on Colab: Runtime → Restart) and run the notebook again from the top.

---

# Part 1: An image is a grid of numbers

## 🟢 Information: A first image

`scikit-image` includes a small set of sample pictures in `skimage.data`, so we can start without downloading anything.
(See [the documentation](https://scikit-image.org/docs/stable/api/skimage.data.html) for the source and licence of each.)

Here is a photograph of a cat called "Chelsea" ... not to be confused here with the eponymous district of London ;).

`plt.imshow` shows an image; `plt.axis("off")` hides the axes; `plt.show()` finishes the figure.

In [ ]:
cat = data.chelsea()

plt.imshow(cat)
plt.axis("off")
plt.show()

## 🟢 New topic: arrays

To Python, that photograph is an **array**: a grid of numbers, from the `numpy` package.
You can think of it as a list of lists (notebook b), but with extra tools, and a fixed size.

An array has a **shape** (its dimensions) and a **dtype** (the type of number it stores).

In [ ]:
print(type(cat))
print("shape:", cat.shape)
print("dtype:", cat.dtype)

Read the
- shape as **(rows, columns, channels)**:
    - 300 **rows**: the image is 300 pixels tall.
    - 451 **columns**: the image is 451 pixels wide.
    - 3 **channels**: red, green and blue (RGB). Each pixel is three numbers.
- dtype `uint8` as "unsigned 8-bit integer":
    - unsigned means no positive vs negative
    - integer means whole numbers
    - 8-bit is what's requires for $2^8 = 256$ elements, here from 0 (none of that colour) to 255 (the most).

To see it, here is an image so small we can write it by hand, as a list of lists of lists.
It has 2 rows and 2 columns, so four pixels.

In [ ]:
tiny = np.array([
    [[255, 0, 0], [0, 255, 0]],
    [[0, 0, 255], [255, 255, 255]],
], dtype=np.uint8)

print(tiny.shape)
plt.imshow(tiny)
plt.axis("off")
plt.show()

---
### 🟣 Core Exercise: Read a tiny image

1. *Before* running anything, predict in the Markdown cell below: what colour is the pixel `tiny[0, 1]`? And `tiny[1, 0]`?
2. Print both and check.
3. Make the bottom-right pixel yellow (red and green at full, no blue), and display `tiny` again.

*Your predictions and answer here.*

---
---
---

### ✅ Solution: Read a tiny image

In [ ]:
print(tiny[0, 1])
print(tiny[1, 0])

tiny[1, 1] = [255, 255, 0]
plt.imshow(tiny)
plt.axis("off")
plt.show()

`tiny[0, 1]` is `[0 255 0]`: row 0, column 1, which is green. `tiny[1, 0]` is `[0 0 255]`: blue.
Yellow is red plus green, so the bottom-right pixel `[255, 255, 0]` shows as yellow.

📝 Note the order: **row first, then column**.
That is (y, x), the *opposite* of the (x, y) you may expect from graphs.
This catches nearly everyone out at least once.

---
## 🟢 Information: Indexing and slicing

Everything from notebook b carries over, with one index per dimension, separated by commas.
- `cat[100, 200]` is one pixel: row 100, column 200.
- `cat[:, :, 0]` is *every* row, *every* column, channel 0 (red).
- `cat[50:200, 150:350]` is a **slice**: rows 50 up to (not including) 200, columns 150 up to 350.
  A slice of an image is a **crop**.

In [ ]:
print(cat[100, 200])

red = cat[:, :, 0]
print("red channel shape:", red.shape)

eyes = cat[50:200, 150:350]
print("crop shape:", eyes.shape)
plt.imshow(eyes)
plt.axis("off")
plt.show()

---
### 🟣 Core Exercise: Crop the nose

1. Display the full image *with its axes* (leave out `plt.axis("off")`), and use the tick labels to find the rows and columns that contain the cat's nose.
2. Crop to just the nose with a slice, print its shape, and display it.
3. In the Markdown cell below, say how you chose the numbers, and which came first in your slice: rows or columns?

*Your answer here.*

---
---
---

### ✅ Solution: Crop the nose

In [ ]:
plt.imshow(cat)       # no axis("off"): we want the numbers on the axes
plt.show()

nose = cat[215:265, 220:300]
print(nose.shape)
plt.imshow(nose)
plt.axis("off")
plt.show()

Your numbers will differ a little: any slice that shows the nose is fine.
From the axes we read the nose at roughly rows 225 to 255 and columns 230 to 290, and added a margin.
The *vertical* axis gives the rows, so rows come first: `cat[rows, columns]`.

---
## 🟢 Information: Numbers about a whole image

Arrays can answer questions about all their pixels at once, with no loop.
`.mean()` averages everything. Adding `axis=(0, 1)` averages over rows and columns, leaving one number per channel.

In [ ]:
print("overall mean:", cat.mean())
print("mean per channel:", cat.mean(axis=(0, 1)))

for index, name in enumerate(["red", "green", "blue"]):
    print(f"{name:6} {cat[:, :, index].mean():6.1f}")

The cat is more red than green, and more green than blue: orange-brown fur, as expected.

But RGB is awkward for the questions we will ask, such as "how *vivid* is this?" or "how much *purple* is there?".
No single RGB channel answers either.
Part 2 switches to a representation designed for those questions.

---

# Part 2: Measuring colour

## 🟢 Information: Hue, saturation and value

The **HSV** representation describes each pixel with three numbers, each from 0 to 1:
- **Hue**: *which* colour, as a position round the colour wheel (0 is red, then yellow, green, cyan, blue, magenta, and back to red at 1).
- **Saturation**: *how vivid*. 0 is grey; 1 is the purest form of the colour.
- **Value**: *how bright*. 0 is black.

`color.rgb2hsv` converts. Its output has the same shape, but each channel is now hue, saturation, value (as decimals, not whole numbers).

In [ ]:
hsv = color.rgb2hsv(cat)
hue, saturation, value = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]
print(hsv.shape, hsv.dtype)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].imshow(cat)
axes[0].set_title("original")
axes[1].imshow(saturation, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("saturation (white = vivid)")
axes[2].imshow(value, cmap="gray", vmin=0, vmax=1)
axes[2].set_title("value (white = bright)")
for ax in axes:
    ax.axis("off")
plt.show()

Notice that the white fur and the background are low in saturation (dark in the middle panel),
while the orange and green are high.

## 🟢 Information: Booleans meet arrays

A comparison on an array makes a Boolean array: one `True` or `False` per pixel.
This is called a **mask** (notebook c, applied to every pixel at once).

Because `True` counts as 1 and `False` as 0, the `.mean()` of a mask is the **share** of pixels that are `True`.

In [ ]:
vivid = saturation > 0.5
print(vivid.shape, vivid.dtype)
print(f"share of vivid pixels: {vivid.mean():.3f}")

plt.imshow(vivid, cmap="gray")
plt.axis("off")
plt.show()

📝 `0.5` is a **decision** we made.
"Vivid" has no single correct threshold, and a different cut-off gives a different number.
Whenever a measurement is turned into a category, a line has been chosen,
either by someone, or some algorithmic process.
Try adapting the `0.5` to see what difference it makes.

## 🟢 Information: Three measures in functions

We now package the steps as functions (notebook e), so we can apply them to many images.

- `to_hsv` converts, and copes with two common cases: a grey image (2 dimensions, no channels) and an image with a fourth "alpha" (transparency) channel.
  `image[..., :3]` means "all rows, all columns, the first three channels".
- `centre_crop` cuts away a margin. Some images (like fashion plates) have paper borders and captions.
We typically do not want to measure them.
- `purple_mask` marks pixels whose hue falls in a purple range.
  We chose it by looking at the colour wheel below, and we check it there.
- `colour_summary` returns four numbers for an image.

In [ ]:
PURPLE_HUE = (0.72, 0.88)   # a range on the 0-1 hue wheel: blue-violet to magenta
VIVID = 0.5                 # saturation above this counts as "vivid"


def to_hsv(image):
    if image.ndim == 2:                 # grey image: make it 3-channel
        image = color.gray2rgb(image)
    return color.rgb2hsv(image[..., :3])  # drop any alpha channel


def centre_crop(image, margin=0.2):
    rows, columns = image.shape[:2]
    top, left = int(rows * margin), int(columns * margin)
    return image[top:rows - top, left:columns - left]


def purple_mask(hsv):
    hue, sat, val = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    in_range = (hue >= PURPLE_HUE[0]) & (hue <= PURPLE_HUE[1])
    return in_range & (sat > 0.25) & (val > 0.3)


def colour_summary(image):
    hsv = to_hsv(image)
    sat, val = hsv[..., 1], hsv[..., 2]
    return {
        "brightness": val.mean(),
        "saturation": sat.mean(),
        "vivid_share": (sat > VIVID).mean(),
        "purple_share": purple_mask(hsv).mean(),
    }

📝 `&` is the array version of `and` (notebook c): it combines two masks pixel by pixel.
We need brackets round each comparison.

**Check a measure where you already know the answer.**
`data.colorwheel()` is a picture of the whole hue wheel. If `purple_mask` works, it should select only the purple wedge.

In [ ]:
wheel = data.colorwheel()
mask = purple_mask(to_hsv(wheel))

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(wheel)
axes[0].set_title("colour wheel")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title("purple_mask")
for ax in axes:
    ax.axis("off")
plt.show()

The mask picks out a wedge on the purple-to-magenta side, as intended.
This gives us more confidence to use it on "unseen" images where we *don't* know the answer in advance.

---
### 🟣 Core Exercise: Predict, then check

Here are three photographs from `skimage.data`: `data.chelsea()` (the cat), `data.astronaut()` and `data.coffee()`.

1. Display all three.
2. *Before* computing anything, predict in the Markdown cell below which has the **highest** `saturation` and which the **lowest**. Make a prediction for `brightness` too.
3. Run `colour_summary` on each and print the results (rounded to 3 places).
4. Compare with your predictions. Where were you wrong, and why?

*Your predictions and comparison here.*

---
---
---

### ✅ Solution: Predict, then check

In [ ]:
test_images = {
    "cat": data.chelsea(),
    "astronaut": data.astronaut(),
    "coffee": data.coffee(),
}

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, (name, image) in zip(axes, test_images.items()):
    ax.imshow(image)
    ax.set_title(name)
    ax.axis("off")
plt.show()

for name, image in test_images.items():
    summary = colour_summary(image)
    print(f"{name:10}", {key: round(float(number), 3) for key, number in summary.items()})

With `scikit-image` 0.26 we get:

| | brightness | saturation | vivid_share | purple_share |
|---|---|---|---|---|
| cat | 0.579 | 0.432 | 0.337 | 0.000 |
| astronaut | 0.561 | 0.357 | 0.375 | 0.012 |
| coffee | 0.622 | 0.725 | 0.861 | 0.000 |

- `coffee` is by far the most saturated (the photo is dominated by strongly coloured objects) and the `astronaut` the least.
- Brightness is surprisingly similar across all three: a reminder that "bright" and "colourful" are different things.
- `vivid_share` and `saturation` do not rank the cat and astronaut the same way (0.432 vs 0.357 for the mean, but 0.337 vs 0.375 for the share).
  Two sensible measures of the same idea can disagree, so *say which one you used*.

(Here `zip(axes, test_images.items())` pairs each panel with an image, and `(name, image)` unpacks each pair. A handy pattern, but not one we need beyond this.)

---
## 🟢 Information: Why not average the hue?

We measured brightness and saturation by averaging, but we did not average the *hue*. Why not?
Hue is a position on a **circle**: 0.99 and 0.01 are both red, and only 0.02 apart.
Their ordinary mean is 0.5, which is cyan, the opposite colour.

In [ ]:
print(np.mean([0.99, 0.01]))

# Three pixels, in hue-saturation-value: two reds and a cyan
swatches = color.hsv2rgb(np.array([[[0.99, 1, 1], [0.01, 1, 1], [0.5, 1, 1]]]))
plt.imshow(swatches)
plt.axis("off")
plt.show()

📝 So the measure of "how much purple" counts pixels *in a hue range* (a share, via a mask), rather than averaging hues.
Choosing a measure that suits the *kind* of number you have is part of the work.

On to a real collection.

---

# Part 3: A real collection

## 🟢 Information: The V&A and its fashion plates

The [Victoria and Albert Museum](https://www.vam.ac.uk) in London holds over a million objects,
and offers a free, open [API](https://developers.vam.ac.uk) (a way for programs to request records) for the collection.
Each record has details such as title, date and place, and a link to an image.
We will search for **fashion plates**, one decade at a time, from the 1820s to the 1890s:
this span encompasses a temporal range from before to after 1856,
with roughly the same range on either side.

⚠️ **Before you download anything:**
- Review the [API](./apis_json.ipynb) notebook. Some notes:
- **Rights.** V&A images are offered mainly for non-commercial, educational and research use, and rights vary by object.
Read the V&A [terms of use](https://www.vam.ac.uk/articles/terms-of-use) yourself.
This notebook downloads images *while it runs*, for your own study; nothing is stored in the repository.
- **Be polite to servers.**
We send an identifying `User-Agent`, pause between requests, and download each image *once* (later runs reuse the saved copy).
- **Do not commit** downloaded images to a public repository unless you have checked that you may.
- **Record** where everything came from.
Below we save a table of exactly which objects we used.

📝 This is a **sample, not the whole collection**: a few plates per decade, chosen by the search's own ranking.
That ranking is a decision, with consequences (which we will return to below).

The cell below defines `fetch`: it requests a web address and returns the response as bytes.

In [ ]:
USER_AGENT = "Student using the python-in-the-smoke notebook (https://markgotham.github.io/python-in-the-smoke)"


def fetch(url):
    request = Request(url, headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=30) as response:
        return response.read()

## 🟢 Information: One search, and what comes back

We ask the API for five records matching the exact phrase "fashion plate" (the quotation marks mean *that phrase*, not either word),
made between 1860 and 1869, and with an image.
`urlencode` turns a dictionary of settings into the right form for a web address.
The response is JSON, which `json.loads` turns into ordinary dictionaries and lists (notebooks b and apis_json).

In [ ]:
VAM_SEARCH = "https://api.vam.ac.uk/v2/objects/search"

params = {
    "q": '"fashion plate"',
    "year_made_from": 1860,
    "year_made_to": 1869,
    "images_exist": "true",
    "page_size": 5,
}
response = json.loads(fetch(VAM_SEARCH + "?" + urlencode(params)))

print(response["info"]["record_count"], "matching records in total")
print(len(response["records"]), "returned")

record = response["records"][0]
print(record["systemNumber"])
print(record.get("_primaryTitle"))
print(record.get("_primaryDate"))
print(record["_images"]["_iiif_image_base_url"])

The image address is built from that **base** plus a standard ending:
`full/!400,400/0/default.jpg` means the whole image (`full`),
fitted inside 400 by 400 pixels (`!400,400`), unrotated (`0`), in default quality.
This is the IIIF standard
for images ([V&A notes here](https://developers.vam.ac.uk/images)).
We ask for a small size on purpose: it is quicker, kinder to the server, and plenty enough for our colour question.

---
### 🟣 Core Exercise: Read the records

1. For all five records in `response["records"]`, print the title, date, place (`_primaryPlace`) and object type (`objectType`).
Use `.get(...)` so that a missing field gives `None` instead of an error.
2. In the Markdown cell below, write what you notice. Are they all what you expected? Is anything missing or oddly formatted?

*Your observations here.*

---
---
---

### ✅ Solution: Read the records

In [ ]:
for record in response["records"]:
    print(
        record.get("_primaryTitle"), "|",
        record.get("_primaryDate"), "|",
        record.get("_primaryPlace"), "|",
        record.get("objectType"),
    )

The records come from a live database, so your output may differ from anyone else's, and from tomorrow's.
Things worth looking for:
- **Dates** are text, not numbers, and may be a single year, a range, or a phrase with extra words.
- Some **fields are empty**: not every record has a place or a maker.
- **Object types** may include things other than what you expected from the search.
- Titles may be a name of a magazine or a description, and say nothing about the clothes.

That is normal for museum data, and it shapes what we can measure.
Always look at the raw records before building on them.

---
## 🟢 Information: Build a sample

For each decade from 1820 to 1890 (a `range`, with a step of 10) we take the first few results of a search.
The function `search_decade` does one decade and returns a list of dictionaries, one per usable record; a record with no image is skipped.
The loop collects them all into one `pandas` table, pausing between requests.

📝 "First few results" is reproducible, but not random:
the search puts what it judges the best matches first, which may favour well-catalogued objects.
That arguably suits our purposes well;
for another research question on representation, it may not be so suitable.

In [ ]:
DECADES = list(range(1820, 1900, 10))
PER_DECADE = 8


def search_decade(decade, how_many=PER_DECADE):
    params = {
        "q": '"fashion plate"',
        "year_made_from": decade,
        "year_made_to": decade + 9,
        "images_exist": "true",
        "page_size": how_many,
    }
    response = json.loads(fetch(VAM_SEARCH + "?" + urlencode(params)))
    rows = []
    for record in response["records"]:
        base_url = (record.get("_images") or {}).get("_iiif_image_base_url")
        if base_url is None:
            continue                      # no usable image: skip
        maker = record.get("_primaryMaker") or {}
        rows.append({
            "decade": decade,
            "system_number": record["systemNumber"],
            "title": record.get("_primaryTitle"),
            "date": record.get("_primaryDate"),
            "place": record.get("_primaryPlace"),
            "object_type": record.get("objectType"),
            "maker": maker.get("name"),
            "base_url": base_url,
        })
    return rows


try:
    all_rows = []
    for decade in DECADES:
        all_rows.extend(search_decade(decade))
        time.sleep(0.5)
except Exception as error:
    print("Could not search the V&A API:", error)
    print("Check your connection and re-run this cell. Parts 1 and 2 did not need it.")
    raise

sample = pd.DataFrame(all_rows)
print(len(sample), "records")
sample[["decade", "title", "date", "place", "object_type"]].head(10)

## 🟢 Information: Check the sample

Before measuring anything we check what we have (data literacy, notebooks h and i).

1. How many records per decade, and what are the object types and places?
2. **Dates** are text. We pull out the first four-digit number with a *regular expression* (`\d{4}` means "four digits") to make a `year` column.
   Then we ask: does that year fall in the decade we searched?

In [ ]:
print(sample["decade"].value_counts().sort_index())
print()
print(sample["object_type"].value_counts(dropna=False))
print()
print(sample["place"].value_counts(dropna=False).head(8))

In [ ]:
def first_year(text):
    match = re.search(r"\d{4}", str(text))
    return int(match.group()) if match else None


sample["year"] = sample["date"].apply(first_year)       # apply runs a function on every value

outside = sample[sample["year"].notna() & (sample["year"] // 10 * 10 != sample["decade"])]
print(len(outside), "of", len(sample), "records have a parsed year outside the decade searched")
outside[["decade", "date", "title"]].head()

Any records shown above are worth a look. A plate dated as a range (say "1855-1865") may turn up in more than one decade's search, or sit under a decade that is not its start year. We have not checked exactly how the API treats ranges: you can test it here.
We keep `decade` as the grouping, and keep the `year` column to remind us the dates are fuzzy.
**Date precision limits what questions this data can answer:** a claim about 1856 specifically would be beyond it.

## 🟢 Information: Save the record, then download the images

Two things for good practice.
- **Save the table** (notebook g). This CSV records exactly which objects you used, so you (or someone else) can repeat the study even if the search results later change.
- **Download each image once** into a folder, and reuse it if it is already there. That keeps you offline-friendly and the server's load light.

`Path` (from `pathlib`) is a tidy way to build file paths: `folder / "name.jpg"` joins them.

In [ ]:
sample.to_csv("vam_fashion_plates_sample.csv", index=False)

IMAGE_FOLDER = Path("vam_images")
IMAGE_FOLDER.mkdir(exist_ok=True)


def download_image(base_url, system_number, size=400):
    path = IMAGE_FOLDER / f"{system_number}.jpg"
    if not path.exists():                       # download once only
        content = fetch(f"{base_url}full/!{size},{size}/0/default.jpg")
        path.write_bytes(content)
        time.sleep(0.3)
    return path


paths = []
for _, row in sample.iterrows():                # iterrows gives one row at a time
    try:
        paths.append(download_image(row["base_url"], row["system_number"]))
    except Exception as error:
        print("Failed:", row["system_number"], error)
        paths.append(None)

sample["path"] = paths
sample = sample.dropna(subset=["path"]).reset_index(drop=True)
print(len(sample), "images ready in", IMAGE_FOLDER)

This takes a minute or so the first time and is instant afterwards.

📝 If some downloads fail, the table keeps only the images that worked, and says how many.
That is a quiet source of bias: if the *failures* are not random, neither is what is left.

## 🟢 Information: Look before you measure

The most valuable check is also the simplest: **look at the images.**
`show_images` displays a list of image files in a grid.

In [ ]:
def show_images(paths, titles, ncols=4):
    nrows = math.ceil(len(paths) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3.8 * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for i in range(len(paths)):
        axes.flat[i].imshow(io.imread(paths[i]))
        axes.flat[i].set_title(str(titles[i])[:30], fontsize=8)
    plt.tight_layout()
    plt.show()


early = sample[sample["decade"] == 1820].head(4)
show_images(list(early["path"]), list(early["date"]))

---
### 🟣 Core Exercise: Predict, then look

1. *Before* looking at more images, predict in the Markdown cell below: will the 1890s plates have a higher mean `saturation` than the 1820s plates? Will they have more purple? Give a reason.
2. Display four plates from the 1890s too (copy the cell above, changing the decade).
3. Compare the two rows by eye, and write down **three things you see that could affect a colour measurement**, besides the clothes.
   (Look at margins, captions, paper tone, and whether every plate is actually coloured.)

*Your predictions and observations here.*

---
---
---

### ✅ Solution: Predict, then look

In [ ]:
late = sample[sample["decade"] == 1890].head(4)
show_images(list(late["path"]), list(late["date"]))

What you see depends on the live sample, but typical things to find are:
- **Paper**: old paper is often yellowed or brown, and that tone is *not* the colour of the dress. It adds saturation to everything.
- **Margins and captions**: black text and blank borders. Our `centre_crop` removes some, but not all.
- **Uncoloured plates**: some plates are black-and-white engravings. Measuring their saturation measures the paper.
- **Several figures per plate, or backgrounds**, so the colours may not be the clothes.
- **Different image sizes and shapes**, and different scanning conditions.

Every item here is a way for a number to change for a reason other than "fashion changed". Keep the list: Part 4 tests some of them.

---

# Part 4: Measure, compare, and try to break the result

## 🟢 Information: Measure every plate

We now apply what we built in Part 2 to each image, in a loop (notebook d), keeping the **centre** of each plate.
The results go into a table next to the details of each object (notebook f).

In [ ]:
measure_rows = []
for path in sample["path"]:
    image = io.imread(path)
    measure_rows.append(colour_summary(centre_crop(image)))

measures = pd.DataFrame(measure_rows)
results = pd.concat([sample, measures], axis=1)
results[["decade", "date", "brightness", "saturation", "vivid_share", "purple_share"]].head()

## 🟢 Information: Compare the decades

First the **averages per decade**, and *how many plates* each average rests on (`size`).
Then a plot showing every plate as a dot (so we see the spread, not just the average) and the decade mean as a line.
The dashed line marks 1856: our decade points are *start* years, so 1856 falls between the 1850s and 1860s.

In [ ]:
measure_names = ["brightness", "saturation", "vivid_share", "purple_share"]

table = results.groupby("decade")[measure_names].mean().round(3)
table["n"] = results.groupby("decade").size()
table

In [ ]:
def plot_measure(results, column):
    decade_means = results.groupby("decade")[column].mean()
    plt.figure(figsize=(7, 3.5))
    plt.scatter(results["decade"], results[column], alpha=0.4, label="single plates")
    plt.plot(decade_means.index, decade_means.values, marker="o", color="black", label="decade mean")
    plt.axvline(1856, linestyle="--", color="grey")
    plt.xlabel("decade searched (start year)")
    plt.ylabel(column)
    plt.legend()
    plt.show()


plot_measure(results, "saturation")
plot_measure(results, "purple_share")

---
### 🟣 Core Exercise: The claim, tested

The claim: *plates became more vivid, and more purple, after 1856.*

1. In one or two sentences, say what the table and plots show. Mention how many plates each point rests on, and how much the single plates spread around the decade mean.
2. Use a mask (notebook c) to split the table into **before 1860** (decades 1820 to 1850) and **1860 onwards**. Print the mean of each measure for the two groups.
   (We split by decade, so the 1850s go in the first group: that is a decision, and it puts some post-1856 plates there.)
3. State, in one sentence, whether the numbers support the claim. Then say what you *cannot* conclude from them.

*Your answer here.*

---
---
---

### ✅ Solution: The claim, tested

In [ ]:
results["period"] = results["decade"].apply(lambda d: "1860 onwards" if d >= 1860 else "before 1860")
results.groupby("period")[measure_names].mean().round(3)

The `lambda d: ...` is a one-line function, written in place (the same as `def` with a `return`).

Your numbers come from the live sample, so we do not quote them here.
Whatever direction they point, a sound answer has this shape:
- Say what *is* in the table: for example "the mean saturation is higher in the later group, by about X, on N plates".
- Look at the spread: if single plates in the early group range as widely as the difference between the group means, the difference is weak evidence.
- With only a few plates per decade, one unusual plate moves a mean a lot.
- **Do not claim** that synthetic dyes *caused* any change: the data has nothing about dyes, only about images.

---
### 🟣 Core Exercise: Five ways this could be wrong

For each of the following, explain **how it could produce a rise in saturation or purple even if dyes changed nothing**, and name **one check** you could make (with data you already have, or could get):

1. Printing technology changed over the century (from hand-colouring to colour printing).
2. Some plates in the sample are uncoloured.
3. The paper and the scans differ.
4. We took the first 8 plates per decade from the search results.
5. A fashion plate shows an ideal, not what people wore.

*Your answers here.*

---
---
---

### ✅ Solution: Five ways this could be wrong

| Threat | How it could fake a trend | A check |
|---|---|---|
| **Printing technology** | Hand-coloured plates can look paler than colour-printed ones. A rise in saturation could be about the *printing*, not the dress. | Group by `maker` (the publisher), or compare plates by place, and see whether the trend holds within one publication. |
| **Uncoloured plates** | Black-and-white plates have low saturation (just paper). If the *share* of them changes by decade, the average changes with it. | Flag near-grey plates (say, `saturation` below some cut-off you choose) and count them per decade. Then repeat the comparison without them. |
| **Paper and scans** | Yellowed paper adds saturation; lighting varies. | Measure a patch of blank margin, using a slice such as `image[:20, :20]` (⭐ below), and see if the *paper* trend matches the *plate* trend. |
| **First 8 from the search** | The search's ranking may favour well-catalogued or well-photographed items, which may differ by decade. | Increase `PER_DECADE`, or take page 2 (`page=2` in the search), and see if the result holds. |
| **An ideal, not reality** | Plates show what a magazine wanted to sell, not what most people wore. | None, in this data: this limits what the *question* can mean. A study of actual garments (also in the V&A collection) would ask something different. |

The first four can be tested in code. The fifth cannot: it is a limit on the claim itself.

A study that lists and tests its own weaknesses is more convincing than one with a bigger sample and no checks.

---
### ⭐ Optional Exercise: Measure the paper

Take the table's `path` column and, for every plate, measure the mean saturation of a small patch of blank corner (`image[:20, :20]`), compared with the `centre_crop`.
Add it to `results` as a new column and plot it by decade, as with `plot_measure`.
Does the paper's colour trend like the plates' colour does?

*Your answer here.*

---
---
---

### ✅ Solution: Measure the paper

In [ ]:
paper_saturation = []
for path in results["path"]:
    image = io.imread(path)
    corner = image[:20, :20]                       # top-left 20 by 20 pixels
    paper_saturation.append(to_hsv(corner)[..., 1].mean())

results["paper_saturation"] = paper_saturation
plot_measure(results, "paper_saturation")

If the paper's saturation follows the same pattern as the plates', then a good part of our "trend" may be about paper and scanning conditions.
If it does not, that is some reassurance (though not proof).
Either way, the corner patch itself might include a stamp, a caption or a stain: look at a few.

---
## 🟢 Information: Does the measure catch what we meant?

A different check on the measure: look at the plates it scores *highest* for purple.
`nlargest(4, "purple_share")` returns the four rows with the largest values.

In [ ]:
top_purple = results.nlargest(4, "purple_share")
print(top_purple[["decade", "date", "purple_share"]])
show_images(list(top_purple["path"]), [f"{share:.2f} purple" for share in top_purple["purple_share"]])

---
### 🟣 Core Exercise: Is it a purple dress?

Look at the four plates above. For each, decide which of these it is, and write one line of reasoning:
- **Right**: there is purple clothing, and that is what the measure found.
- **Wrong thing**: the purple is something else (background, stain, print, a different hue read as purple).
- **Unsure**: you cannot tell from the image.

How many of the four were *right*? Then say what that suggests about how far to trust `purple_share`.

*Your answers here.*

---
---
---

### ✅ Solution: Is it a purple dress?

There is no fixed answer because the sample is live, but this is the same idea as *precision* in the NLP notebook: of what the measure *flagged*, how much is what we *meant*?
- If most are purple clothing, the measure is doing its job on this sample.
- If some are backgrounds, pink or lilac-toned paper, or a faded or foxed print, the measure picks up other things, and a rise in `purple_share` could be about those.
- Even all four being right would say little about what it *misses* (recall): purple clothing the measure does not flag, such as a very pale lilac or a dark violet below our brightness cut-off.

To do this properly, you would label a larger, *random* sample by hand and count, as for entities.

---
### ⭐ Optional Exercise: A palette portrait

`segmentation.slic` groups neighbouring, similar-coloured pixels into regions (**superpixels**). Replacing each region with its average colour gives a "flattened" version of an image, showing what colours it is made of.
Try it on the first plate in `results` (or on `data.chelsea()`), and display original and flattened side by side.
Change `n_segments` and see what happens.

*Your observations here.*

---
---
---

### ✅ Solution: A palette portrait

In [ ]:
image = io.imread(results.loc[0, "path"])
regions = segmentation.slic(image, n_segments=100, start_label=1)
flattened = color.label2rgb(regions, image, kind="avg")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(image)
axes[1].imshow(flattened)
for ax in axes:
    ax.axis("off")
plt.show()

Fewer segments gives a coarser, more abstract palette; more segments is closer to the original.
This kind of picture is useful to *look* at, and each region's colour is also a number you could measure.
(`slic` can also be fussy about greyscale images and very small ones, so it is no replacement for checking.)

---

# Part 5: Your own project

## 🟢 Information: What you just did, as a template

| Step | In this notebook |
|---|---|
| 1. A question, in a form data could inform | Did fashion plates get more vivid or purple after 1856? |
| 2. A collection that could hold the answer | V&A fashion plates, via its API |
| 3. A sample, with its limits stated | First 8 per decade, 1820 to 1890 |
| 4. Download politely, and record provenance | Polite downloads; saved CSV of objects used |
| 5. Look at the data | Contact sheet; checked records and dates |
| 6. Measure, and check the measure | HSV measures; colour wheel; purple dresses check |
| 7. Compare, and try to break the result | Decades; five threats; paper patch |

Each step is a place where a project can go wrong, and a place where *your* choices make it yours.

## 🟢 Information: Project ideas

Each of these can be started with the tools you now have. None has a settled answer here.

| Question | Where the images could come from | A first measure | Watch out for |
|---|---|---|---|
| Does the **same fashion-plate question** hold for *Paris* vs *London* plates? | The same V&A search; `place` column (already in your table) | The same measures, grouped by place | Different magazines, printing and sample sizes by place |
| How has **poster colour** changed over the 20th century? | V&A and other poster collections (check each one's terms) | Saturation, brightness, `vivid_share` by decade | Very different sizes and print methods; poster text and white borders |
| Did **wallpaper or textile** palettes change over a century or two? | V&A, searching for `wallpaper` or `textile` with year filters | Hue shares (for example, share of green) | Fading, scanning, and what survives to be collected |
| How **dark or hazy** do painters make London? | Public-domain painting scans from museums and Wikimedia Commons | Brightness, and its spread (contrast) | Varnish and yellowing, differences in photography of paintings |
| How are **maps** coloured? (For example, Charles Booth's 1889 map colours London streets by social class.) | Library and archive map collections (check terms: rights vary) | The share of each map colour, by area | Faded ink, scanning, and whose categories the colours encode |
| Can we prepare **scanned print** for text recognition (OCR)? | Digitised newspapers and books (copyright varies by item) | Thresholding to separate ink from paper (⭐ below) | OCR errors become your data; the same scan problems as above |

📝 Also consider people. Images of living people raise privacy and consent issues that fashion plates and posters do not. Think before analysing photographs of individuals.

## 🟢 Information: Other tools in scikit-image

You have used a small corner of `scikit-image`. Others, by task (all in the [documentation](https://scikit-image.org/docs/stable/)):
- **Edges and shapes:** `filters.sobel`, `feature.canny`.
- **Separating foreground from background:** `filters.threshold_otsu`, and `segmentation`.
- **Measuring regions:** `measure.label` and `measure.regionprops` (area, position, shape of each region).
- **Texture:** `feature.local_binary_pattern`.
- **Finding a known pattern in a larger image:** `feature.match_template`.

For *most* questions in the arts and humanities, the harder part is not the algorithm. It is the question, the collection, and the checks above.

### ⭐ Optional Exercise: Ink and paper

A scanned printed page is a good test case. `data.page()` is a small greyscale scan.
`filters.threshold_otsu` picks a brightness cut-off that separates dark ink from light paper automatically.
Apply it, and display the page and the result (a mask) side by side. Is every letter preserved?
The mask is what an OCR program would work from.

*Your observations here.*

---
---
---

### ✅ Solution: Ink and paper

In [ ]:
page = data.page()
cutoff = filters.threshold_otsu(page)
ink_on_paper = page > cutoff           # True for pixels lighter than the cutoff

print("cutoff:", cutoff)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].imshow(page, cmap="gray")
axes[1].imshow(ink_on_paper, cmap="gray")
for ax in axes:
    ax.axis("off")
plt.show()

With `scikit-image` 0.26 the cutoff is 157.
A single cut-off for the whole page struggles where the lighting is uneven: the left side of this scan is dark, and much of it goes black.
Such images need a *local* threshold (`filters.threshold_local`), which adapts to each neighbourhood.
Our deliberately imperfect first attempt shows *why* checking matters.

## 🟢 Information: Before you start

The same questions apply to any image project:
- **Where did the images come from?** Which collection, which scan, which date, and under what terms?
- **What does the measure actually measure?** Check it on a case where you know the answer.
- **Did you look?** At the images, and at the extremes.
- **Who or what is missing from the sample?**
- **What is your record?** A saved table of objects, the code, and the package versions (printed in the set up).

---
## ⚠️ Common Errors

| Symptom | What to check |
|---|---|
| `ModuleNotFoundError: No module named 'skimage'` | Install with `%pip install scikit-image`, then restart the kernel. The install name and the import name differ |
| `SyntaxError` on `import scikit-image` | Use `import skimage` (or `from skimage import io`). A hyphen is a minus sign to Python |
| `ValueError: the input array must have size 3 along channel_axis` | The image is grey (2 dimensions) or has an alpha channel (4). Use `to_hsv`, or `image[..., :3]` |
| `IndexError: index ... is out of bounds` | Rows come first. Check `image.shape` and whether you swapped rows and columns |
| Colours look wrong after adding or multiplying | `uint8` values wrap round past 255. Convert first (see below) |
| `FileNotFoundError` when reading an image | Check the path, and which folder the notebook is running in (notebook g) |
| `HTTPError 403` or `429`, `URLError`, or a timeout | Check your connection. Send a `User-Agent`, slow down, and download once. The site may have changed |
| `KeyError: '_images'` (or similar) in API results | Some records lack the field. Use `.get()`, as in Part 3 |
| An empty table or fewer images than expected | The search may return few results, or some downloads may have failed: print `len(...)` at each step |
| Different results from someone else | Compare package versions (printed in the set up), and remember that the V&A data is live and can change |

**The cells below fail on purpose.**
Read each message, work out the cause, then compare with the explanation underneath.

### ⚠️ Deliberate error 1: the wrong package name

In [ ]:
import scikit-image

`SyntaxError: invalid syntax`: Python reads `scikit-image` as `scikit` **minus** `image`.
The package is *installed* as `scikit-image` but *imported* as `skimage`.

### ⚠️ Deliberate error 2: rows and columns

In [ ]:
cat = data.chelsea()
print(cat.shape)
cat[600, 10]

`IndexError: index 600 is out of bounds for axis 0 with size 300`: the image has only 300 rows (axis 0).
The first index is the row, so `600` is too large. A crop meant for a 451-wide image may need the numbers the other way round.

### ⚠️ Deliberate error 3: a grey image where colour is expected

In [ ]:
grey = data.camera()
print(grey.shape)
color.rgb2hsv(grey)

`ValueError: the input array must have size 3 along channel_axis, got (512, 512)`: a grey image has no channel dimension, and neither do some scans.
Our `to_hsv` function fixes this with `color.gray2rgb`. The same error with `(..., 4)` means an alpha (transparency) channel, which `image[..., :3]` removes.

### ⚠️ Deliberate error 4: an error that is *not* an error

In [ ]:
print(np.array([200, 100], dtype=np.uint8) + 100)

brighter = cat + 100        # intended: a brighter picture
plt.imshow(brighter)
plt.axis("off")
plt.show()

**No error message appears**, which is what makes this the most dangerous.
`uint8` cannot hold more than 255, so 200 + 100 *wraps round* to 44, and bright pixels turn dark and strangely coloured.
To brighten safely, convert to a type with room, add, and clip:
`np.clip(cat.astype(int) + 100, 0, 255).astype(np.uint8)`.
Always look at the result of an image operation.

### ⚠️ Deliberate error 5: a file that is not there

In [ ]:
io.imread("vam_images/does_not_exist.jpg")

`FileNotFoundError: [Errno 2] No such file or directory: ...`: the path is wrong, or the download never happened.
Check the folder (notebook g) and print `IMAGE_FOLDER` and the list of files if in doubt.

---
## 🟢 Information: Summary

| To ... | Write ... |
|---|---|
| Install and import | `%pip install scikit-image`, then `from skimage import io, color, data` |
| Load a built-in image | `data.chelsea()`, `data.astronaut()`, ... |
| Read an image file | `io.imread(path)` |
| Show an image | `plt.imshow(image)`, `plt.axis("off")`, `plt.show()` |
| Size and type | `image.shape` (rows, columns, channels), `image.dtype` |
| A pixel, a channel, a crop | `image[r, c]`, `image[:, :, 0]`, `image[r1:r2, c1:c2]` |
| Convert to hue, saturation, value | `color.rgb2hsv(image)` |
| A mask and a share | `mask = sat > 0.5`; `mask.mean()` |
| Combine masks | `mask1 & mask2` |
| Average a whole image | `image.mean()`; per channel `image.mean(axis=(0, 1))` |
| Search a museum API | `json.loads(fetch(url + "?" + urlencode(params)))` |
| Group and compare | `results.groupby("decade")[columns].mean()` |
| Split an image into regions | `segmentation.slic(...)`, `color.label2rgb(..., kind="avg")` |
| Separate ink from paper | `filters.threshold_otsu(page)` |

A measurement from an image is a claim to test.
Look at the images, check the measure where you know the answer, say how big and how chosen your sample was, and list the ways the result could be wrong.

**Final reflection:**
- What did you *expect* before the measurements, and what did you find?
- Which of the five threats worried you most, and could you test it?
- What decisions did you make that someone else might make differently (a threshold, a crop, a decade split)?
- What question about images from London's museums and archives would *you* want to explore, and what would you need to check first?

### Source and documentation

- [scikit-image documentation](https://scikit-image.org/docs/stable/) and [sample data](https://scikit-image.org/docs/stable/api/skimage.data.html)
- [V&A Collections API](https://developers.vam.ac.uk/), [image notes](https://developers.vam.ac.uk/images) and [terms of use](https://www.vam.ac.uk/articles/terms-of-use)
- [matplotlib documentation](https://matplotlib.org/stable/)

The sample images in Parts 1, 2 and 5 are part of `scikit-image` (see its documentation for each one's source and licence).
No museum images are included in this repository: Part 3 shows how to download them, and what to check first.

End

---